<a href="https://colab.research.google.com/github/sushmitajha-commits/Growth-Pod-Trackers/blob/main/SDR_Payout_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests psycopg2-binary pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 14.9 MB/s eta 0:00:00


In [ ]:
import requests
import psycopg2
import pandas as pd
from datetime import datetime, timedelta

API_KEY = "NzZlZTk0YzctY2Q5OS00YjRjLTkxYWYtODMxYjk3NGJlZDdi"
WORKSPACE_ID = "63defca333d43c53913f648f"

DB_CONFIG = {
    "host": "gw-rds-analytics.celzx4qnlkfp.us-east-1.rds.amazonaws.com",
    "database": "gw_prod",
    "user": "airbyte_user",
    "password": "airbyte_user_password",
    "port": 5432
}

In [ ]:
from datetime import timezone

now = datetime.now(timezone.utc)
start = datetime(2026, 1, 1)  # safe floor to capture all historical data
end = now

print(f"Pulling data from {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")

Pulling data from 2026-01-01 to 2026-04-27


In [ ]:
url = f"https://reports.api.clockify.me/v1/workspaces/63defca333d43c53913f648f/reports/detailed"

headers = {
    "X-Api-Key": "NzZlZTk0YzctY2Q5OS00YjRjLTkxYWYtODMxYjk3NGJlZDdi",
    "Content-Type": "application/json"
}

all_entries = []
page = 1
page_size = 500
max_pages = 500

while page <= max_pages:
    payload = {
        "dateRangeStart": start.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
        "dateRangeEnd": end.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
        "detailedFilter": {
            "page": page,
            "pageSize": page_size
        }
    }

    response = requests.post(url, json=payload, headers=headers)
    data = response.json()
    entries = data.get("timeentries", [])

    if not entries:
        print(f"No more entries at page {page}. Stopping.")
        break

    all_entries.extend(entries)
    print(f"Page {page}: fetched {len(entries)} entries (total so far: {len(all_entries)})")

    if len(entries) < page_size:
        print("Last page reached (partial page).")
        break

    page += 1

if page > max_pages:
    print(f"WARNING: Hit max_pages cap ({max_pages}). There may be more data.")

print(f"\nTotal entries fetched: {len(all_entries)}")

Page 1: fetched 500 entries (total so far: 500)
Page 2: fetched 500 entries (total so far: 1000)
Page 3: fetched 500 entries (total so far: 1500)
Page 4: fetched 500 entries (total so far: 2000)
Page 5: fetched 500 entries (total so far: 2500)
Page 6: fetched 500 entries (total so far: 3000)
Page 7: fetched 500 entries (total so far: 3500)
Page 8: fetched 500 entries (total so far: 4000)
Page 9: fetched 500 entries (total so far: 4500)
Page 10: fetched 500 entries (total so far: 5000)
Page 11: fetched 500 entries (total so far: 5500)
Page 12: fetched 500 entries (total so far: 6000)
Page 13: fetched 500 entries (total so far: 6500)
Page 14: fetched 500 entries (total so far: 7000)
Page 15: fetched 333 entries (total so far: 7333)
Last page reached (partial page).

Total entries fetched: 7333


In [ ]:
rows = []

for e in all_entries:
    rows.append({
        "entry_id": e.get("_id"),
        "user_id": e.get("userId"),
        "user_name": e.get("userName"),
        "user_email": e.get("userEmail"),
        "project_id": e.get("projectId"),
        "project_name": e.get("projectName"),
        "client_id": e.get("clientId"),
        "client_name": e.get("clientName"),
        "task_name": e.get("taskName"),
        "start_time": e["timeInterval"]["start"],
        "end_time": e["timeInterval"].get("end"),
        "duration_seconds": e["timeInterval"].get("duration"),
        "billable": e.get("billable"),
        "is_locked": e.get("isLocked"),
        "currency": e.get("currency"),
        "amount": e.get("amount"),
        "rate": e.get("rate"),
        "earned_amount": e.get("earnedAmount")
    })

df = pd.DataFrame(rows)

# Verification
print(f"Total rows: {len(df)}")
if 'start_time' in df.columns and len(df) > 0:
    df['start_time'] = pd.to_datetime(df['start_time'])
    print(f"Date range: {df['start_time'].min()} to {df['start_time'].max()}")
    print(f"\nEarliest 5 entries:")
    print(df.nsmallest(5, 'start_time')[['user_name', 'project_name', 'start_time', 'duration_seconds']])

Total rows: 7333
Date range: 2026-01-01 01:30:00+05:30 to 2026-04-27 10:55:48+05:30

Earliest 5 entries:
            user_name project_name                start_time  duration_seconds
7332  Camille Poyaoan     External 2026-01-01 01:30:00+05:30              9031
7331     Gayatri Pant     Internal 2026-01-01 07:05:47+05:30              8419
7330     Gayatri Pant     Internal 2026-01-01 09:34:43+05:30             12238
7329     Gayatri Pant     Internal 2026-01-01 13:41:00+05:30              3639
7328     Gayatri Pant     Internal 2026-01-01 15:15:00+05:30              8049


In [ ]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

insert_query = """
INSERT INTO gist.clockify_time_entries (
    entry_id, user_id, user_name, user_email,
    project_id, project_name, client_id, client_name,
    task_name, start_time, end_time, duration_seconds,
    billable, is_locked, currency, amount, rate, earned_amount
)
VALUES (
    %(entry_id)s, %(user_id)s, %(user_name)s, %(user_email)s,
    %(project_id)s, %(project_name)s, %(client_id)s, %(client_name)s,
    %(task_name)s, %(start_time)s, %(end_time)s, %(duration_seconds)s,
    %(billable)s, %(is_locked)s, %(currency)s, %(amount)s, %(rate)s, %(earned_amount)s
)
ON CONFLICT (entry_id) DO UPDATE SET
    duration_seconds = EXCLUDED.duration_seconds,
    end_time = EXCLUDED.end_time;
"""

for row in rows:
    cur.execute(insert_query, row)

conn.commit()
cur.close()
conn.close()